#MobileNetV2_Encoder

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def conv_bn_relu6(in_ch, out_ch, k=3, s=1, p=1, groups=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, groups=groups, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU6(inplace=True),
    )

def conv_bn(in_ch, out_ch, k=1, s=1, p=0, groups=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, groups=groups, bias=False),
        nn.BatchNorm2d(out_ch),
    )

class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio):
        super().__init__()
        assert stride in [1, 2]
        hidden = int(round(in_ch * expand_ratio))
        self.use_res = (stride == 1 and in_ch == out_ch)

        layers = []
        if expand_ratio != 1:
            layers.append(conv_bn_relu6(in_ch, hidden, k=1, s=1, p=0))
        layers.append(conv_bn_relu6(hidden, hidden, k=3, s=stride, p=1, groups=hidden))  # depthwise
        layers.append(conv_bn(hidden, out_ch, k=1, s=1, p=0))  # project
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        out = self.block(x)
        return x + out if self.use_res else out

class MobileNetV2_Encoder(nn.Module):
    """
    MobileNetV2 paper backbone (feature extractor).
    Returns multi-scale features for DeepLab/MFA.
    """
    def __init__(self, width_mult=1.0):
        super().__init__()

        def c(ch):  # width multiplier
            return int(ch * width_mult)

        # (t, c, n, s)
        cfg = [
            (1, 16, 1, 1),
            (6, 24, 2, 2),
            (6, 32, 3, 2),
            (6, 64, 4, 2),
            (6, 96, 3, 1),
            (6, 160, 3, 2),
            (6, 320, 1, 1),
        ]

        self.stem = conv_bn_relu6(3, c(32), k=3, s=2, p=1)

        layers = []
        in_ch = c(32)

        self.stage_out_ids = []  # indices of layers we may want as feature maps
        idx = 0
        for (t, out_ch, n, s) in cfg:
            out_ch = c(out_ch)
            for i in range(n):
                stride = s if i == 0 else 1
                layers.append(InvertedResidual(in_ch, out_ch, stride=stride, expand_ratio=t))
                in_ch = out_ch
                idx += 1
            self.stage_out_ids.append(idx - 1)  # last block index of this stage

        self.blocks = nn.ModuleList(layers)
        self.conv_last = conv_bn_relu6(in_ch, c(1280), k=1, s=1, p=0)

    def forward(self, x):
        x = self.stem(x)  # 112x112

        feats = []
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            # Save end-of-stage features (paper stages)
            if i in self.stage_out_ids:
                feats.append(x)

        deep = self.conv_last(x)

        # feats correspond to: c16, c24, c32, c64, c96, c160, c320 (end of each stage)
        # For segmentation you often want:
        # shallow1 = feats[1] (c24, 56x56)
        # shallow2 = feats[2] (c32, 28x28)
        # shallow3 = feats[3] (c64, 14x14)
        # deep     = deep    (c1280, 7x7)
        shallow1 = feats[1]
        shallow2 = feats[2]
        shallow3 = feats[3]
        return shallow1, shallow2, shallow3, deep




#Atrous Convolution

In [2]:
def atrous_conv(in_ch, out_ch, rate):
    return nn.Sequential(
        nn.Conv2d(
            in_ch, out_ch,
            kernel_size=3,
            stride=1,
            padding=rate,
            dilation=rate,
            bias=False
        ),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True)
    )


#ASPP

In [3]:
class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch=256):
        super().__init__()

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

        self.branch2 = atrous_conv(in_ch, out_ch, rate=3)
        self.branch3 = atrous_conv(in_ch, out_ch, rate=6)
        self.branch4 = atrous_conv(in_ch, out_ch, rate=12)
        self.branch5 = atrous_conv(in_ch, out_ch, rate=24)

        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

        self.project = nn.Sequential(
            nn.Conv2d(out_ch * 6, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        h, w = x.shape[2], x.shape[3]

        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        b5 = self.branch5(x)

        bp = self.global_pool(x)
        bp = F.interpolate(bp, size=(h, w), mode="bilinear", align_corners=False)

        x = torch.cat([b1, b2, b3, b4, b5, bp], dim=1)
        return self.project(x)


#SE Module

In [4]:
import torch
import torch.nn as nn

class SEModule(nn.Module):
    def __init__(self, channels: int, r: int = 16):
        super().__init__()
        reduced = max(channels // r, 1)

        self.pool = nn.AdaptiveAvgPool2d(1)          # [B,C,H,W] -> [B,C,1,1]
        self.fc1  = nn.Conv2d(channels, reduced, 1)  # like FC
        self.relu = nn.ReLU(inplace=True)
        self.fc2  = nn.Conv2d(reduced, channels, 1)
        self.sig  = nn.Sigmoid()

    def forward(self, x):
        w = self.pool(x)
        w = self.fc1(w)
        w = self.relu(w)
        w = self.fc2(w)
        w = self.sig(w)
        return x * w


# Encoder


In [5]:
import torch
import torch.nn as nn

class CleanEncoder(nn.Module):


    def __init__(self, backbone: nn.Module, aspp: nn.Module, se: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.aspp = aspp
        self.se = se

    def forward(self, x: torch.Tensor):
        # Safety check so you NEVER pass feature maps by mistake
        if x.dim() != 4 or x.size(1) != 3:
            raise ValueError(
                f"CleanEncoder expects an image tensor [B,3,H,W]. "
                f"Got {tuple(x.shape)}"
            )

        # Backbone outputs
        s1, s2, s3, deep = self.backbone(x)

        # Deep branch: ASPP -> SE
        deep_aspp = self.aspp(deep)
        deep_se   = self.se(deep_aspp)

        return s1, s2, s3, deep_se


# Pretrained backbone


In [6]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

class MobileNetV2_Pretrained_Encoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        weights = MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        m = mobilenet_v2(weights=weights)

        self.features = m.features

        # Layer indices verified for torchvision mobilenet_v2
        self.s1_idx = 3   # 24 channels, 1/4
        self.s2_idx = 6   # 32 channels, 1/8
        self.s3_idx = 10  # 64 channels, 1/16
        self.deep_idx = 18  # 1280 channels, 1/32

    def forward(self, x):
        s1 = s2 = s3 = None

        for i, layer in enumerate(self.features):
            x = layer(x)

            if i == self.s1_idx:
                s1 = x
            elif i == self.s2_idx:
                s2 = x
            elif i == self.s3_idx:
                s3 = x

        deep = x  # after layer 18

        return s1, s2, s3, deep

#MFF, SA, ESA, GPA and the full workflow

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# -------------------------
# outputs concat channels: 32 + 48 + 64 + 128 = 272
# -------------------------
class MFF(nn.Module):
    def __init__(
        self,
        c1=24, c2=32, c3=64,           # from your encoder s1/s2/s3
        out1=32, out2=48, out3=64,     # recommended from your file
        se_r=16
    ):
        super().__init__()
        self.conv1 = nn.Conv2d(c1, out1, 1, bias=False)
        self.conv2 = nn.Conv2d(c2, out2, 1, bias=False)
        self.conv3 = nn.Conv2d(c3, out3, 1, bias=False)

        self.bn1 = nn.BatchNorm2d(out1)
        self.bn2 = nn.BatchNorm2d(out2)
        self.bn3 = nn.BatchNorm2d(out3)

        self.se1 = SEModule(out1, r=se_r)
        self.se2 = SEModule(out2, r=se_r)
        self.se3 = SEModule(out3, r=se_r)

    def forward(self, s1, s2, s3, deep_up):
        # s1: [B,24,56,56] reference
        f1 = F.relu(self.bn1(self.conv1(s1)), inplace=True)

        # s2: [B,32,28,28] -> upsample to 56x56
        f2 = F.relu(self.bn2(self.conv2(s2)), inplace=True)
        f2 = F.interpolate(f2, size=f1.shape[-2:], mode="bilinear", align_corners=False)

        # s3: [B,64,14,14] -> upsample to 56x56
        f3 = F.relu(self.bn3(self.conv3(s3)), inplace=True)
        f3 = F.interpolate(f3, size=f1.shape[-2:], mode="bilinear", align_corners=False)

        # SE on shallow features (paper)
        f1 = self.se1(f1)
        f2 = self.se2(f2)
        f3 = self.se3(f3)

        # concat shallow + deep_up
        return torch.cat([f1, f2, f3, deep_up], dim=1)  # [B,272,56,56]

# -------------------------
# GPA (use your GPA code; here is a safe wrapper interface)
# Your GPA expects input channels = 96 and returns 96 (adds residual).
# -------------------------
# If you already have GPA class, import it and delete this block.
class SA(nn.Module):
    def __init__(self, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=(kernel_size-1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        max_pool = torch.max(x, dim=1, keepdim=True)[0]
        att = self.sigmoid(self.conv(torch.cat([avg_pool, max_pool], dim=1)))
        return x * att

class ESA(nn.Module):
    def __init__(self, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)  # [B,C,1,1]
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)

class GPALocal(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = SA()
        self.esa = ESA()

    def forward(self, x):
        r1 = x
        r2 = F.adaptive_avg_pool2d(x, (x.shape[2] // 2, x.shape[3] // 2))
        r3 = F.adaptive_avg_pool2d(x, (x.shape[2] // 4, x.shape[3] // 4))
        r4 = F.adaptive_avg_pool2d(x, (x.shape[2] // 8, x.shape[3] // 8))

        r1 = self.sa(self.esa(r1))
        r2 = self.sa(self.esa(r2))
        r3 = self.sa(self.esa(r3))
        r4 = self.sa(self.esa(r4))

        r2 = F.interpolate(r2, size=x.shape[-2:], mode="bilinear", align_corners=False)
        r3 = F.interpolate(r3, size=x.shape[-2:], mode="bilinear", align_corners=False)
        r4 = F.interpolate(r4, size=x.shape[-2:], mode="bilinear", align_corners=False)

        return torch.cat([r1, r2, r3, r4], dim=1)  # [B, 96*4, H, W]

class GPAGlobal(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = SA()
        self.esa = ESA()

    def forward(self, x):
        return self.sa(self.esa(x))

class GPA(nn.Module):
    def __init__(self, c=96):
        super().__init__()
        self.local = GPALocal()
        self.global_ = GPAGlobal()
        self.reduce_local = nn.Conv2d(c * 4, c, kernel_size=1)

    def forward(self, x):
        local_out = self.reduce_local(self.local(x))
        global_out = self.global_(x)
        return x + local_out + global_out

# -------------------------
# FINAL: Full model wiring (your encoder + ASPP + SE + MFF + GPA + classifier)
# -------------------------
class MFADeepLabV3Plus(nn.Module):
    def __init__(self, backbone, aspp, num_classes: int, dropout_rate: float = 0.5):
        super().__init__()
        self.backbone = backbone     # returns (s1,s2,s3,deep)
        self.aspp = aspp             # deep -> [B,256,h,w]
        self.se_deep = SEModule(256, r=16)

        # Dropout for deep features
        self.dropout_deep = nn.Dropout2d(dropout_rate)

        # deep 1x1 conv to 128 (matches your decoder file idea) :contentReference[oaicite:5]{index=5}
        self.deep_proj = nn.Conv2d(256, 128, kernel_size=1, bias=False)
        self.deep_bn = nn.BatchNorm2d(128)

        self.mff = MFF(c1=24, c2=32, c3=64, out1=32, out2=48, out3=64)

        # After concat (272ch) -> 3x3 conv -> 96ch (matches your decoder file + GPA input)
        self.fuse_conv = nn.Sequential(
            nn.Conv2d(272, 96, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
        )
        # Dropout for fused features
        self.dropout_fused = nn.Dropout2d(dropout_rate)

        self.gpa = GPA(c=96)

        # final classifier
        self.classifier = nn.Conv2d(96, num_classes, kernel_size=1)

    def forward(self, x):
        # 1) backbone
        s1, s2, s3, deep = self.backbone(x)

        # 2) deep path: ASPP -> SE -> Dropout
        deep = self.aspp(deep)            # [B,256, h, w]
        deep = self.se_deep(deep)         # [B,256, h, w]
        deep = self.dropout_deep(deep)    # Apply dropout to deep features

        # 3) deep proj -> upsample to s1 spatial size (56x56 if input 224)
        deep = F.relu(self.deep_bn(self.deep_proj(deep)), inplace=True)   # [B,128,h,w]
        deep_up = F.interpolate(deep, size=s1.shape[-2:], mode="bilinear", align_corners=False)

        # 4) MFF concat (shallow processed + deep_up) -> [B,272,56,56]
        fused = self.mff(s1, s2, s3, deep_up)

        # 5) 3x3 conv -> Dropout -> GPA -> upsample to input -> logits
        y = self.fuse_conv(fused)         # [B,96,56,56]
        y = self.dropout_fused(y)         # Apply dropout to fused features
        y = self.gpa(y)                   # [B,96,56,56]

        # upsample to original image size
        y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)

        return self.classifier(y)         # [B,num_classes,H,W]

In [8]:
import torch, os
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"


CUDA: True
GPU: Tesla T4


In [9]:
from google.colab import drive
!fusermount -u /content/drive 2>/dev/null
!rm -rf /content/drive
drive.mount('/content/drive', force_remount=True)

ROOT = "/content/drive/MyDrive/FoodSeg103"
IMG_TRAIN_DIR = f"{ROOT}/Images/img_dir/train"
ANN_TRAIN_DIR = f"{ROOT}/Images/ann_dir/train"
IMG_TEST_DIR  = f"{ROOT}/Images/img_dir/test"
ANN_TEST_DIR  = f"{ROOT}/Images/ann_dir/test"

TRAIN_LIST = f"{ROOT}/ImageSets/train.txt"
TEST_LIST  = f"{ROOT}/ImageSets/test.txt"

with open(TRAIN_LIST, "r") as f:
    train_files = [l.strip() for l in f if l.strip()]   # includes .jpg
with open(TEST_LIST, "r") as f:
    test_files = [l.strip() for l in f if l.strip()]

print("train:", len(train_files), "test:", len(test_files), "example:", train_files[-1])




Mounted at /content/drive
train: 4983 test: 2135 example: 00006866.jpg


In [10]:
# Make a validation set from the exsistance test set
from sklearn.model_selection import train_test_split
validation_set, new_test_files = train_test_split(
    test_files,
    test_size=2/3,
    random_state=42,
    shuffle=True
)

print("validation:", len(validation_set), "new test:", len(new_test_files))
print("validation:", validation_set[0], validation_set[-1], validation_set[len(validation_set) // 2])

validation: 711 new test: 1424
validation: 00006867.jpg 00005257.jpg 00007104.jpg


In [11]:
import os
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode


IGNORE_INDEX = 255

# Final reduced class names
NEW_CLASSES = [
    "background",         # 0
    "french_fries",       # 1
    "almond",             # 2
    "red_beans",          # 3
    "cashew",             # 4
    "dried_cranberries",  # 5
    "soy",                # 6
    "walnut",             # 7
    "peanut",             # 8
    "egg",                # 9
    "apple",              # 10
    "date",               # 11
    "apricot",            # 12
    "avocado",            # 13
    "banana",             # 14
    "strawberry",         # 15
    "cherry",             # 16
    "blueberry",          # 17
    "raspberry",          # 18
    "mango",              # 19
    "olives",             # 20
    "peach",              # 21
    "lemon",              # 22
    "pear",               # 23
    "fig",                # 24
    "pineapple",          # 25
    "grape",              # 26
    "kiwi",               # 27
    "melon",              # 28
    "orange",             # 29
    "watermelon",         # 30
    "steak",              # 31
    "pork",               # 32
    "chicken_duck",       # 33
    "sausage",            # 34
    "lamb",               # 35
    "seafood",            # 36
    "fish",               # 37
    "shrimp",             # 38
    "bread",              # 39
    "corn",               # 40
    "pasta",              # 41
    "noodles",            # 42
    "rice",               # 43
    "tofu",               # 44
    "eggplant",           # 45
    "potato",             # 46
    "garlic",             # 47
    "cauliflower",        # 48
    "tomato",             # 49
    "seaweed",            # 50
    "spring_onion",       # 51
    "ginger",             # 52
    "okra",               # 53
    "lettuce",            # 54
    "pumpkin",            # 55
    "cucumber",           # 56
    "white_radish",       # 57
    "carrot",             # 58
    "asparagus",          # 59
    "broccoli",           # 60
    "cabbage",            # 61
    "onion",              # 62
    "pepper",             # 63
    "green_beans",        # 64
    "mushroom",           # 65
    "herbs",              # 66
]

NUM_CLASSES = len(NEW_CLASSES)

# Original FoodSeg103 ID -> new reduced ID
ORIGINAL_TO_NEW = {
    0: 0,      # background

    1: 255,    # candy
    2: 255,    # egg tart
    3: 1,      # french fries
    4: 255,    # chocolate
    5: 255,    # biscuit
    6: 255,    # popcorn
    7: 255,    # pudding
    8: 255,    # ice cream
    9: 255,    # cheese butter
    10: 255,   # cake
    11: 255,   # wine
    12: 255,   # milkshake
    13: 255,   # coffee
    14: 255,   # juice
    15: 255,   # milk
    16: 255,   # tea

    17: 2,     # almond
    18: 3,     # red beans
    19: 4,     # cashew
    20: 5,     # dried cranberries
    21: 6,     # soy
    22: 7,     # walnut
    23: 8,     # peanut
    24: 9,     # egg

    25: 10,    # apple
    26: 11,    # date
    27: 12,    # apricot
    28: 13,    # avocado
    29: 14,    # banana
    30: 15,    # strawberry
    31: 16,    # cherry
    32: 17,    # blueberry
    33: 18,    # raspberry
    34: 19,    # mango
    35: 20,    # olives
    36: 21,    # peach
    37: 22,    # lemon
    38: 23,    # pear
    39: 24,    # fig
    40: 25,    # pineapple
    41: 26,    # grape
    42: 27,    # kiwi
    43: 28,    # melon
    44: 29,    # orange
    45: 30,    # watermelon

    46: 31,    # steak
    47: 32,    # pork
    48: 33,    # chicken duck
    49: 34,    # sausage
    50: 255,   # fried meat
    51: 35,    # lamb
    52: 255,   # sauce
    53: 36,    # crab -> seafood
    54: 37,    # fish
    55: 36,    # shellfish -> seafood
    56: 38,    # shrimp
    57: 255,   # soup

    58: 39,    # bread
    59: 40,    # corn
    60: 255,   # hamburg
    61: 255,   # pizza
    62: 255,   # hanamaki baozi
    63: 255,   # wonton dumplings
    64: 41,    # pasta
    65: 42,    # noodles
    66: 43,    # rice
    67: 255,   # pie
    68: 44,    # tofu
    69: 45,    # eggplant
    70: 46,    # potato
    71: 47,    # garlic
    72: 48,    # cauliflower
    73: 49,    # tomato
    74: 50,    # kelp -> seaweed
    75: 50,    # seaweed
    76: 51,    # spring onion
    77: 255,   # rape
    78: 52,    # ginger
    79: 53,    # okra
    80: 54,    # lettuce
    81: 55,    # pumpkin
    82: 56,    # cucumber
    83: 57,    # white radish
    84: 58,    # carrot
    85: 59,    # asparagus
    86: 255,   # bamboo shoots
    87: 60,    # broccoli
    88: 255,   # celery stick
    89: 66,    # cilantro mint -> herbs
    90: 255,   # snow peas
    91: 61,    # cabbage
    92: 255,   # bean sprouts
    93: 62,    # onion
    94: 63,    # pepper
    95: 64,    # green beans
    96: 64,    # French beans -> green_beans
    97: 65,    # king oyster mushroom -> mushroom
    98: 65,    # shiitake -> mushroom
    99: 65,    # enoki mushroom -> mushroom
    100: 65,   # oyster mushroom -> mushroom
    101: 65,   # white button mushroom -> mushroom
    102: 255,  # salad
    103: 255,  # other ingredients
}


def remap_mask(mask_np: np.ndarray, mapping: dict, ignore_index: int = 255) -> np.ndarray:
    """
    mask_np: HxW numpy array with original class IDs
    returns: HxW numpy array with new class IDs
    """
    remapped = np.full(mask_np.shape, ignore_index, dtype=np.uint8)
    for old_id, new_id in mapping.items():
        remapped[mask_np == old_id] = new_id
    return remapped


class FoodSeg103Dataset(Dataset):
    def __init__(self, files, img_dir, ann_dir, size=(256, 256), augment=False):
        self.files = files
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_file = self.files[idx]
        base = os.path.splitext(img_file)[0]

        img_path = os.path.join(self.img_dir, img_file)
        mask_path = os.path.join(self.ann_dir, base + ".png")

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # Resize image and mask
        image = TF.resize(image, self.size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.size, interpolation=InterpolationMode.NEAREST)

        # Optional augmentations
        if self.augment:
            if np.random.rand() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

        # Convert image to tensor
        image = TF.to_tensor(image)

        # Convert mask to numpy, remap, then to tensor
        mask = np.array(mask, dtype=np.uint8)
        mask = remap_mask(mask, ORIGINAL_TO_NEW, ignore_index=IGNORE_INDEX)
        mask = torch.as_tensor(mask, dtype=torch.long)

        return image, mask
# Start small for speed & stability
train_ds = FoodSeg103Dataset(train_files, IMG_TRAIN_DIR, ANN_TRAIN_DIR, size=(512,512), augment=True)
val_ds   = FoodSeg103Dataset(validation_set, IMG_TEST_DIR, ANN_TEST_DIR, size=(512,512), augment=False)
test_ds  = FoodSeg103Dataset(new_test_files,  IMG_TEST_DIR,  ANN_TEST_DIR,  size=(512,512), augment=False)

train_loader = DataLoader(train_ds, batch_size=8 if device=="cuda" else 2, shuffle=True, num_workers=2, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=8 if device=="cuda" else 2, shuffle=False, num_workers=2, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=8 if device=="cuda" else 2, shuffle=False, num_workers=2, pin_memory=False)

# sanity
images, masks = next(iter(train_loader))
print(images.shape, images.dtype)
print(masks.shape, masks.dtype, "min/max:", masks.min().item(), masks.max().item())

torch.Size([8, 3, 512, 512]) torch.float32
torch.Size([8, 512, 512]) torch.int64 min/max: 0 255


In [12]:
import os

ckpt_dir = "/content/drive/MyDrive/FoodSeg103/67-class pretrainedCheckpoints"
os.makedirs(ckpt_dir, exist_ok=True)

print("Checkpoint directory ready:", ckpt_dir)

Checkpoint directory ready: /content/drive/MyDrive/FoodSeg103/67-class pretrainedCheckpoints


# Loss Metrics

In [13]:
# =========================
# CE + Dice Loss (Multiclass)
# Drop-in replacement for `criterion`
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    """
    Multiclass Dice Loss computed from logits.

    logits: [B, C, H, W]
    target: [B, H, W] with class ids in [0, C-1]
    """
    def __init__(self, num_classes: int, smooth: float = 1.0, ignore_index=None):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # logits: [B,C,H,W]
        # target: [B,H,W]
        if target.dtype != torch.long:
            target = target.long()

        # Softmax -> probabilities per class
        probs = F.softmax(logits, dim=1)  # [B,C,H,W]

        # Handle ignore_index by masking it out (optional)
        if self.ignore_index is not None:
            mask = (target != self.ignore_index)  # [B,H,W]
            # Replace ignored targets with 0 temporarily so one_hot works
            target = target.clone()
            target[~mask] = 0
        else:
            mask = None

        # One-hot encode target: [B,H,W] -> [B,H,W,C] -> [B,C,H,W]
        target_1h = F.one_hot(target, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        # Apply mask if ignore_index is used
        if mask is not None:
            mask_f = mask.unsqueeze(1).float()  # [B,1,H,W]
            probs = probs * mask_f
            target_1h = target_1h * mask_f

        # Compute Dice per class (micro-average over batch+spatial)
        dims = (0, 2, 3)  # sum over batch, H, W
        intersection = torch.sum(probs * target_1h, dims)
        union = torch.sum(probs, dims) + torch.sum(target_1h, dims)

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)  # [C]
        dice_loss = 1.0 - dice  # [C]

        # Mean over classes
        return dice_loss.mean()


class CombinedCELossDice(nn.Module):
    """
    criterion = CE + lambda_dice * Dice
    """
    def __init__(self, num_classes: int, lambda_dice: float = 1.0, ignore_index=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index) if ignore_index is not None else nn.CrossEntropyLoss()
        self.dice = DiceLoss(num_classes=num_classes, ignore_index=ignore_index)
        self.lambda_dice = lambda_dice

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce_loss = self.ce(logits, target)
        dice_loss = self.dice(logits, target)
        return ce_loss + self.lambda_dice * dice_loss




### 1. Initialize Model, Loss Function, and Optimizer

In [14]:
num_classes = NUM_CLASSES # Use the global NUM_CLASSES which is 67
backbone = MobileNetV2_Pretrained_Encoder(pretrained=True)
# Initialize the model and move it to the selected device
model = MFADeepLabV3Plus(backbone= backbone, aspp=ASPP(1280, 256), num_classes=num_classes, dropout_rate=0.5)
model.to(device)


# As per user clarification, all pixels belong to a class (0-103), so no ignore_index is needed.
criterion = CombinedCELossDice(num_classes=num_classes, lambda_dice=1.0, ignore_index=255)

# Define the Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)

print(f"Model initialized on {device} with {num_classes} classes.")
print("Loss function: CrossEntropyLoss")
print("Optimizer: AdamW")

Model initialized on cuda with 67 classes.
Loss function: CrossEntropyLoss
Optimizer: AdamW


# Latest Model

In [15]:
import os
import torch
import json

ckpt_dir = "/content/drive/MyDrive/FoodSeg103/pretrainedCheckpoints-3"
os.makedirs(ckpt_dir, exist_ok=True)

ckpt_latest_path = os.path.join(ckpt_dir, "latest.pth")
history_path = os.path.join(ckpt_dir, "history.json")

start_epoch = 0
global_step = 0 # Initialize global_step here
# best_val_loss = float("inf")
# best_val_miou = -1.0 # Initialize for mIoU based saving
patience_counter = 0
current_epoch_train_miou = 0.0 # Initialize to hold the latest computed train mIoU
best_val_fwiou = -1.0 # Initialize for FWIoU based saving

# ---- Load checkpoint if exists ----
if os.path.exists(ckpt_latest_path):
    print("🔁 Loading checkpoint...")
    ckpt = torch.load(ckpt_latest_path, map_location=device)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])

    start_epoch = ckpt["epoch"] + 1
    val_loss = ckpt.get("val_loss", float("inf"))
    val_miou = ckpt.get("val_miou", -1.0)
    val_fwiou = ckpt.get("val_fwiou", -1.0)
    global_step = ckpt.get("global_step", 0) # Load global_step from checkpoint

    print(f"✅ Resuming from epoch {start_epoch}, global step {global_step}")
    print(f"   loss so far: {val_loss:.4f}")
    print(f"   mIoU so far: {val_miou:.4f}") # P
    print(f"   FWIoU so far: {val_fwiou:.4f}") # P
else:
    print("🆕 No checkpoint found, starting from scratch")
    global_step = 0 # Explicitly set to 0 if starting fresh

# ---- Load or create history ----
if os.path.exists(history_path):
    with open(history_path, "r") as f:
        history = json.load(f)
    train_losses = history.get("train_losses", [])
    val_losses   = history.get("val_losses", [])
    val_mious    = history.get("val_mious", []) # Initialize val_mious list
    train_mious  = history.get("train_mious", []) # Initialize train_mious list
    val_fwiou_history = history.get("val_fwiou_history", []) # Initialize val_fwiou_history list
    print(f"📈 Loaded history with {len(train_losses)} epochs")
else:
    train_losses = []
    val_losses = []
    val_mious = []
    train_mious = [] # Initialize train_mious list
    val_fwiou_history = [] # Initialize val_fwiou_history list
    print("📈 No history found, starting fresh")

🆕 No checkpoint found, starting from scratch
📈 No history found, starting fresh


# Best Model

In [16]:
import os
import torch

# Define the path to the best model checkpoint
best_model_path = os.path.join(ckpt_dir, "best_model.pth")

if os.path.exists(best_model_path):
    print(f"Loading best model from {best_model_path}")
    # Load the checkpoint
    best_ckpt = torch.load(best_model_path, map_location=device)


    # Update global best_val_loss and best_val_miou for training continuity
    global best_val_loss, best_val_miou, best_val_fwiou
    best_val_loss = best_ckpt["best_val_loss"]
    best_val_miou = best_ckpt["best_val_miou"]
    best_val_fwiou = best_ckpt.get("best_val_fwiou", -1.0)


    print("✅ Best model loaded successfully and set to evaluation mode.")
    print(f"   Epoch at saving: {best_ckpt['epoch'] + 1}")
    print(f"   Best validation loss: {best_val_loss:.4f}")
    print(f"   Best validation mIoU: {best_val_miou:.4f}")
    print(f"   Best validation FWIoU: {best_val_fwiou:.4f}")
    if "train_miou_at_best_val_miou" in best_ckpt:
        print(f"   Train mIoU at best val mIoU: {best_ckpt['train_miou_at_best_val_miou']:.4f}")
else:
    best_val_miou = -1.0     # Initialize best_val_miou
    best_val_loss = float("inf")
    best_val_fwiou = -1.0
    print(f"❌ No best model found at {best_model_path}. Please train the model first.")

❌ No best model found at /content/drive/MyDrive/FoodSeg103/pretrainedCheckpoints-3/best_model.pth. Please train the model first.


In [17]:
# import os
# import torch

# # Define the path to the best model checkpoint
# best_model_path = os.path.join(ckpt_dir, "best_model.pth")

# if os.path.exists(best_model_path):
#     print(f"Loading best model from {best_model_path}")
#     # Load the checkpoint
#     best_ckpt = torch.load(best_model_path, map_location=device)


#     # Update global best_val_loss and best_val_miou for training continuity
#     global best_val_loss, best_val_miou
#     best_val_loss = best_ckpt["best_val_loss"]
#     best_val_miou = best_ckpt["best_val_miou"]


#     print("✅ Best model loaded successfully and set to evaluation mode.")
#     print(f"   Epoch at saving: {best_ckpt['epoch'] + 1}")
#     print(f"   Best validation loss: {best_val_loss:.4f}")
#     print(f"   Best validation mIoU: {best_val_miou:.4f}")
#     if "train_miou_at_best_val_miou" in best_ckpt:
#         print(f"   Train mIoU at best val mIoU: {best_ckpt['train_miou_at_best_val_miou']:.4f}")
# else:
#     best_val_miou = -1.0     # Initialize best_val_miou
#     best_val_loss = float("inf")
#     print(f"❌ No best model found at {best_model_path}. Please train the model first.")

# Effective Metrics


In [18]:
import torch
import numpy as np
from torch.amp import autocast
@torch.no_grad()
def fast_confusion_matrix(pred, target, num_classes):
    pred = pred.view(-1)
    target = target.view(-1)

    k = (target >= 0) & (target < num_classes)
    pred = pred[k]
    target = target[k]

    idx = target * num_classes + pred
    cm = torch.bincount(idx, minlength=num_classes**2).reshape(num_classes, num_classes)
    return cm

def segmentation_metrics_from_cm(cm, eps=1e-7):
    tp = torch.diag(cm).float()
    fp = cm.sum(0).float() - tp
    fn = cm.sum(1).float() - tp

    # IoU per class
    iou = tp / (tp + fp + fn + eps)

    # mIoU
    miou = iou.mean().item()

    # 🔥 NEW: FWIoU
    freq = cm.sum(1).float() / (cm.sum() + eps)
    fwiou = (freq * iou).sum().item()

    return miou, fwiou, iou.cpu().numpy()

@torch.no_grad()
def evaluate(model, loader, criterion, num_classes, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    cm_total = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)
    use_amp = (device == "cuda")
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        with autocast(device_type="cuda", enabled=use_amp):
          outputs = model(images)              # [N,C,H,W]
          loss = criterion(outputs, masks)
        total_loss += loss.item()
        n_batches += 1

        preds = outputs.argmax(dim=1)        # [N,H,W]
        cm_total += fast_confusion_matrix(preds, masks, num_classes)

    avg_loss = total_loss / max(1, n_batches)

    miou, fwiou, per_class_iou = segmentation_metrics_from_cm(cm_total)
    return avg_loss, miou, fwiou, per_class_iou


# Training Loop

In [ ]:
import math
import torch
from torch.amp import autocast, GradScaler
import json
import os # Ensure os is imported here

epochs = 100
power = 0.9
initial_lr = 0.003

scaler = GradScaler("cuda")

total_steps = epochs * len(train_loader)

def set_poly_lr(optimizer, base_lr, step, total_steps, power=0.9):
    t = min(step, total_steps - 1) / total_steps # no negetive LR
    lr = base_lr * (1 - step / total_steps) ** power
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr


for epoch in range(start_epoch, epochs):

  #Freezing backbone for the first 10 epochs, so it woudln't destroy the gradiant
    if epoch % 10 == 0:
      for p in model.backbone.parameters():
        p.requires_grad = False
    else:
      for p in model.backbone.parameters():
        p.requires_grad = True

    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        # step-based poly LR
        current_lr = set_poly_lr(optimizer, initial_lr, global_step, total_steps, power)

        with autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        train_loss += loss.item()
        global_step += 1

        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx+1}/{len(train_loader)} "
                  f"| LR {current_lr:.6f} | Loss {train_loss/(batch_idx+1):.4f}")

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    print(f"Epoch {epoch+1} finished. Avg Train Loss: {avg_train_loss:.4f}")

    # -------- VALIDATION --------
    avg_val_loss, val_miou, val_fwiou, _ = evaluate(model, val_loader, criterion, num_classes, device)

    val_losses.append(avg_val_loss)
    val_mious.append(val_miou)
    val_fwiou_history.append(val_fwiou)
    print(
      f"Epoch {epoch+1} | "
      f"Val Loss: {avg_val_loss:.4f} | "
      f"mIoU: {val_miou:.4f} | "
      f"FWIoU: {val_fwiou:.4f}"
    )

    # -------- CHECKPOINTING --------
    improved = val_miou > best_val_miou

    if improved:
        best_val_miou = val_miou
        best_val_loss = avg_val_loss # Keep best_val_loss updated as well
        # patience_counter = 0 # Removed early stopping
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "best_val_miou": best_val_miou,
            "best_val_fwiou": val_fwiou,
            "global_step": global_step,
        }, os.path.join(ckpt_dir, "best_model.pth"))
        print("✅ Improved! Saved best model based on mIoU.")


    if (epoch + 1) % 5 == 0:
        model.eval() # Set model to eval mode for consistent mIoU calculation
        # Removed ignore_index=255 for consistency with criterion initialization
        _, calculated_train_miou, _, _ = evaluate(model, train_loader, criterion, num_classes, device)
        current_epoch_train_miou = calculated_train_miou # Update with the newly calculated value
        train_mious.append(current_epoch_train_miou)
        model.train() # Set model back to train mode
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_loss": avg_val_loss,
            "val_miou": val_miou,
            "val_fwiou": val_fwiou,
            "train_miou": current_epoch_train_miou, # Save the latest available train mIoU
        }, os.path.join(ckpt_dir, f"epoch_{epoch+1:03d}.pth"))
        print(f"Epoch {epoch+1} | Train mIoU: {current_epoch_train_miou:.4f}")
        print(f"💾 Saved model checkpoint for epoch {epoch+1}.")

    # always save latest
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_loss": best_val_loss, # Update with potentially new best_val_loss
        "val_miou": best_val_miou, # Update with potentially new best_val_miou
        "val_fwiou": val_fwiou,
        "global_step": global_step, # Add global_step to the latest checkpoint save
     }, ckpt_latest_path) # Corrected to ckpt_latest_path from ckpt_path

    # Save history after each epoch
    with open(history_path, "w") as f:
        json.dump({
            "train_losses": train_losses,
            "val_losses": val_losses,
            "val_mious": val_mious,
            "train_mious": train_mious,
            "val_fwiou_history": val_fwiou_history
        }, f)
    print("📊 History saved to history.json")

Epoch 1/100 | Batch 100/623 | LR 0.002996 | Loss 2.7860
Epoch 1/100 | Batch 200/623 | LR 0.002991 | Loss 2.5819
Epoch 1/100 | Batch 300/623 | LR 0.002987 | Loss 2.5112
Epoch 1/100 | Batch 400/623 | LR 0.002983 | Loss 2.4487
Epoch 1/100 | Batch 500/623 | LR 0.002978 | Loss 2.4013
Epoch 1/100 | Batch 600/623 | LR 0.002974 | Loss 2.3598
Epoch 1 finished. Avg Train Loss: 2.3517
Epoch 1 | Val Loss: 2.0070 | mIoU: 0.0996 | FWIoU: 0.5750
✅ Improved! Saved best model based on mIoU.
📊 History saved to history.json
Epoch 2/100 | Batch 100/623 | LR 0.002969 | Loss 2.5146
Epoch 2/100 | Batch 200/623 | LR 0.002964 | Loss 2.4532
Epoch 2/100 | Batch 300/623 | LR 0.002960 | Loss 2.4188
Epoch 2/100 | Batch 400/623 | LR 0.002956 | Loss 2.3985
Epoch 2/100 | Batch 500/623 | LR 0.002951 | Loss 2.3781
Epoch 2/100 | Batch 600/623 | LR 0.002947 | Loss 2.3547
Epoch 2 finished. Avg Train Loss: 2.3502
Epoch 2 | Val Loss: 2.1883 | mIoU: 0.0830 | FWIoU: 0.5248
📊 History saved to history.json
Epoch 3/100 | Batch 10